In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_20_try_0.pickle

In [ ]:
%%PandasProfile
### cell 21 ###

question_of_interest_cell33 = 'For how many years have you been writing code and/or programming?'

# Standardize column names and response categories
responses_df_2018_cell10.rename(
    columns={'How long have you been writing code to analyze data?': question_of_interest_cell33},
    inplace=True
)
mapping_2018 = {
    '< 1 year': '< 1 years',
    'I have never written code but I want to learn': 'I have never written code',
    'I have never written code and I do not want to learn': 'I have never written code',
    '1-2 years': '1-3 years',
    '20-30 years': '20+ years',
    '30-40 years': '20+ years',
    '40+ years': '20+ years'
}
responses_df_2018_cell10[question_of_interest_cell33] = (
    responses_df_2018_cell10[question_of_interest_cell33]
    .replace(mapping_2018)
)

responses_df_2019_cell10.rename(
    columns={'How long have you been writing code to analyze data (at work or at school)?': question_of_interest_cell33},
    inplace=True
)
mapping_1_2_to_1_3 = {'1-2 years': '1-3 years'}
# Apply the same 1–2→1–3 years mapping to both 2019 and 2020
for df in (responses_df_2019_cell10, responses_df_2020):
    df[question_of_interest_cell33] = (
        df[question_of_interest_cell33]
        .replace(mapping_1_2_to_1_3)
    )

def combine_subset_of_data_from_multiple_years_33(question, x_axis_title, include_2017=None):
    # Build a single concatenated DataFrame with a 'year' column
    frames = []
    if include_2017 is not None:
        frames.append(responses_df_2017[[question]].assign(year='2017'))
    for year, df in [
        ('2018', responses_df_2018_cell10),
        ('2019', responses_df_2019_cell10),
        ('2020', responses_df_2020),
        ('2021', responses_df_2021),
        ('2022', responses_df_2022_cell10)
    ]:
        frames.append(df[[question]].assign(year=year))
    combined = pd.concat(frames, ignore_index=True)

    # Compute value counts per year and derive percentages
    counts = combined.groupby('year')[question].value_counts(dropna=False)
    totals = combined.groupby('year')[question].count()
    percentages = (counts.div(totals, level='year') * 100).round(1)

    # Format the result
    result = (
        percentages.rename('percentage')
        .reset_index()
        .rename(columns={question: x_axis_title})
    )
    return result

# Combine, sort, and display
title_for_x_axis_cell33 = ''
programming_df_combined = (
    combine_subset_of_data_from_multiple_years_33(
        question_of_interest_cell33,
        title_for_x_axis_cell33
    )
    .sort_values(['year', 'percentage'], ascending=True)
)
programming_df_combined.info()